# 04 — KPI Framework, Metrics Scoring & Insurer Benchmarking
## Indian Health Insurance Statistics — IRDAI Handbook 2021-22

**Purpose:** Define, compute, and score the KPIs that convert descriptive statistics into decision-support intelligence. This notebook produces:
1. A formal KPI register with calculation logic
2. Composite insurer performance scores
3. Market concentration analysis (HHI)
4. Coverage equity index by state
5. Dashboard product specification

**Audience:** IRDAI Policy Division, Head of Actuarial Analytics


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi':120, 'font.family':'DejaVu Sans',
    'axes.spines.top':False, 'axes.spines.right':False,
    'axes.titlesize':13, 'axes.labelsize':11,
})
C_PUBLIC='#1f4e79'; C_PRIVATE='#2e75b6'; C_STANDALONE='#f4b942'
C_ACCENT='#e05c5c'; C_GREEN='#5cb85c'
SECTOR_COLORS = {'Public':C_PUBLIC,'Private':C_PRIVATE,'Standalone':C_STANDALONE}

BASE = '/home/claude/clean_data/'
master    = pd.read_csv(BASE+'master_insurer.csv')
df62      = pd.read_csv(BASE+'t62_health_volume_long.csv')
df66      = pd.read_csv(BASE+'t66_health_claims_long.csv')
df71      = pd.read_csv(BASE+'t71_state_health.csv')
market_total = pd.read_csv(BASE+'market_totals.csv')

master_ins = master[~master['insurer_type'].str.contains('Total', na=False)].copy()
df62_ins   = df62[~df62['insurer_type'].str.contains('Total', na=False)].copy()
df66_ins   = df66[~df66['insurer_type'].str.contains('Total', na=False)].copy()

YEARS = ['2013-14','2014-15','2015-16','2016-17','2017-18','2018-19','2019-20','2020-21','2021-22']
print("✓ Data loaded")


## 4.1 KPI Register

| # | KPI | Business Meaning | Formula | Unit | Limitation |
|---|-----|-----------------|---------|------|------------|
| 1 | Gross Premium CAGR | Market growth velocity | ((P_end/P_start)^(1/n))-1 | % | Nominal; inflation not adjusted |
| 2 | Persons Covered CAGR | Coverage expansion rate | ((C_end/C_start)^(1/n))-1 | % | Count-based; not coverage-depth weighted |
| 3 | Claims Ratio | Underwriting efficiency | Net Claims / Net Earned Premium | % | Does not include expense ratio |
| 4 | Premium Per Person | Pricing level / coverage depth | Gross Premium / Persons Covered | ₹/person/yr | Mix effect from segment composition |
| 5 | Market Concentration (HHI) | Competitive intensity | Σ(share_i²) × 10000 | HHI points | DOJ threshold: >2500 = high concentration |
| 6 | Geographic Gini | Coverage equity | Lorenz-derived inequality measure | 0–1 | Based on premium; not insured population |
| 7 | Underwriting Margin | Profitability signal | 100 - Claims Ratio | % | Excludes opex, commissions, reinsurance |
| 8 | Premium Growth Efficiency | Growth quality | Premium CAGR / Persons CAGR | ratio | >1 = pricing power; <1 = volume-led growth |
| 9 | Sector Shift Index | Competitive disruption | Δmarket share Public vs Standalone | pp | Directional; 3-yr rolling |
| 10 | Claims Settlement Rate | Customer outcome proxy | Claims Settled / Claims Lodged | % | Available only from T76 (limited years) |


In [ ]:
# ── KPI 1 & 2: CAGR computations ─────────────────────────────────────────
def safe_cagr(v0, v1, n):
    try:
        if v0 > 0 and v1 > 0: return ((v1/v0)**(1/n)-1)*100
    except: pass
    return np.nan

# Market-level
m14 = market_total[market_total.year == '2013-14'].iloc[0]
m22 = market_total[market_total.year == '2021-22'].iloc[0]

kpi1_prem_cagr    = safe_cagr(m14['Gross_Premium_Lakh'], m22['Gross_Premium_Lakh'], 8)
kpi2_persons_cagr = safe_cagr(m14['Persons_Covered_000s'], m22['Persons_Covered_000s'], 8)
kpi8_growth_efficiency = kpi1_prem_cagr / kpi2_persons_cagr if kpi2_persons_cagr else np.nan

print(f"KPI 1 — Gross Premium CAGR (8yr):        {kpi1_prem_cagr:.2f}%")
print(f"KPI 2 — Persons Covered CAGR (8yr):      {kpi2_persons_cagr:.2f}%")
print(f"KPI 8 — Premium Growth Efficiency:        {kpi8_growth_efficiency:.2f}x")
print()
print("Interpretation:")
print(f"  Premium growing {kpi8_growth_efficiency:.1f}x faster than coverage — pricing power signal.")
print("  Regulators should monitor whether this reflects benefit enrichment or inflationary pricing.")


In [ ]:
# ── KPI 5: Herfindahl-Hirschman Index (HHI) ──────────────────────────────
def compute_hhi(values):
    total = values.sum()
    if total == 0: return np.nan
    shares = (values / total) * 100  # In percentage
    return (shares**2).sum()

hhi_by_year = {}
df62_total = df62_ins[df62_ins.segment == 'Total'].copy()

for yr in YEARS:
    yr_data = df62_total[
        (df62_total.year == yr) & 
        (df62_total.metric == 'Gross_Premium_Lakh')
    ]['value'].dropna()
    if len(yr_data) > 0:
        hhi_by_year[yr] = compute_hhi(yr_data)

hhi_series = pd.Series(hhi_by_year)

fig, ax = plt.subplots(figsize=(12, 5))
yr_short = ['FY14','FY15','FY16','FY17','FY18','FY19','FY20','FY21','FY22']
hhi_vals = [hhi_series.get(yr, np.nan) for yr in YEARS]
ax.plot(yr_short[:len(hhi_vals)], hhi_vals, marker='o', color=C_PUBLIC, linewidth=2.5, markersize=8)
ax.axhline(2500, color=C_ACCENT, linestyle='--', linewidth=1.5, label='DOJ High Concentration Threshold (2500)')
ax.axhline(1500, color='orange', linestyle='--', linewidth=1.5, label='DOJ Moderate Threshold (1500)')
ax.fill_between(yr_short[:len(hhi_vals)], 0, hhi_vals, alpha=0.1, color=C_PUBLIC)
ax.set_title('KPI 5: Market Concentration (HHI) — Health Insurance Premium
DOJ: <1500=Competitive, 1500-2500=Moderate, >2500=High', fontweight='bold')
ax.set_ylabel('HHI Score')
ax.legend(fontsize=9)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:,.0f}'))
plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/kpi_01_hhi.png', bbox_inches='tight')
plt.show()

latest_hhi = hhi_vals[-1] if hhi_vals else None
print(f"HHI FY2021-22: {latest_hhi:,.0f}")
print(f"HHI FY2013-14: {hhi_vals[0]:,.0f}")
if latest_hhi > 2500:
    print("Classification: HIGH CONCENTRATION — regulatory scrutiny warranted")
elif latest_hhi > 1500:
    print("Classification: MODERATE CONCENTRATION")
else:
    print("Classification: COMPETITIVE MARKET")
print()
print("INSIGHT: HHI above 2500 signals the top few players dominate pricing power.")
print("However, the trend direction matters more: declining HHI = healthy new entrants;")
print("rising HHI = consolidation risk. Cross-reference with standalone insurer growth.")


In [ ]:
# ── KPI 3 + Composite Insurer Performance Score ───────────────────────────
# Score each insurer on 4 dimensions for FY2021-22:
# 1. Premium CAGR (growth, 3yr)
# 2. Claims Ratio (risk — lower is better for profitability, but not always good for policy)
# 3. Persons Covered CAGR (social impact)
# 4. Premium per Person growth (pricing trend — lower growth = more affordable)

# Get 3-yr premium CAGR (FY19-22)
def insurer_cagr(df, yr_start, yr_end, metric='Gross_Premium_Lakh', segment='Total'):
    d = df[(df.segment==segment) & (df.metric==metric)]
    d_start = d[d.year==yr_start].set_index('insurer')['value']
    d_end   = d[d.year==yr_end].set_index('insurer')['value']
    return ((d_end / d_start) ** (1/3) - 1) * 100

cagr_prem  = insurer_cagr(df62_ins, '2019-20', '2021-22', 'Gross_Premium_Lakh')
cagr_pers  = insurer_cagr(df62_ins, '2019-20', '2021-22', 'Persons_Covered_000s')

# Claims ratio FY22 (total segment, aggregate)
cr_22 = master_ins[master_ins.year=='2021-22'].set_index('insurer')['Claims_Ratio_Pct']

# Combine
scorecard = pd.DataFrame({
    'Premium_CAGR_3yr': cagr_prem,
    'Persons_CAGR_3yr': cagr_pers,
    'Claims_Ratio_Pct': cr_22,
}).reset_index().rename(columns={'insurer':'Insurer'})

# Merge insurer type
ins_type = master_ins[master_ins.year=='2021-22'][['insurer','insurer_type']].drop_duplicates()
scorecard = scorecard.merge(ins_type, left_on='Insurer', right_on='insurer', how='left')

# Score (0-10 scale per KPI)
def norm_score(series, higher_better=True, clip_pct=(5,95)):
    lo, hi = series.quantile(clip_pct[0]/100), series.quantile(clip_pct[1]/100)
    s = (series - lo) / (hi - lo + 1e-9) * 10
    s = s.clip(0, 10)
    return s if higher_better else 10 - s

scorecard = scorecard.dropna(subset=['Claims_Ratio_Pct','Premium_CAGR_3yr'])

scorecard['Score_Growth']   = norm_score(scorecard['Premium_CAGR_3yr'], higher_better=True)
scorecard['Score_Coverage'] = norm_score(scorecard['Persons_CAGR_3yr'], higher_better=True)
scorecard['Score_Risk']     = norm_score(scorecard['Claims_Ratio_Pct'], higher_better=False)

# Weighted composite: 35% growth, 30% coverage, 35% risk
scorecard['Composite_Score'] = (
    0.35 * scorecard['Score_Growth'] +
    0.30 * scorecard['Score_Coverage'] +
    0.35 * scorecard['Score_Risk']
)

# Tier assignment
scorecard['Tier'] = pd.cut(scorecard['Composite_Score'],
                            bins=[0,4,6,8,10],
                            labels=['Watch','Developing','Performing','Leader'])

print(scorecard[['Insurer','insurer_type','Premium_CAGR_3yr','Claims_Ratio_Pct',
                  'Composite_Score','Tier']].sort_values('Composite_Score', ascending=False).head(15).to_string())


In [ ]:
# ── Visualize composite scores ────────────────────────────────────────────
sc_plot = scorecard.sort_values('Composite_Score', ascending=False).head(20).copy()
sc_plot['Insurer_Short'] = sc_plot['Insurer'].str.extract(r'^(\w+\s?\w+)')[0]

tier_colors = {'Leader':'#1f4e79','Performing':'#2e75b6','Developing':'#9dc3e6','Watch':'#e05c5c'}
bar_colors = [tier_colors.get(str(t),'gray') for t in sc_plot['Tier']]

fig, ax = plt.subplots(figsize=(13, 7))
bars = ax.barh(sc_plot['Insurer_Short'][::-1], sc_plot['Composite_Score'][::-1],
               color=bar_colors[::-1], edgecolor='white')
ax.axvline(6, color='orange', linestyle='--', linewidth=1.5, alpha=0.8, label='Developing/Performing threshold')
ax.axvline(8, color=C_GREEN, linestyle='--', linewidth=1.5, alpha=0.8, label='Leader threshold')
for bar, score in zip(bars, sc_plot['Composite_Score'][::-1]):
    ax.text(bar.get_width()+0.05, bar.get_y()+bar.get_height()/2,
            f'{score:.1f}', va='center', fontsize=9)

patches = [mpatches.Patch(color=c, label=t) for t,c in tier_colors.items()]
ax.legend(handles=patches, loc='lower right', fontsize=9)
ax.set_xlabel('Composite Performance Score (0-10)')
ax.set_title('Insurer Composite Performance Scorecard (FY2021-22)\nWeighted: 35% Growth + 30% Coverage + 35% Underwriting Risk', fontweight='bold')
ax.set_xlim(0, 11)
plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/kpi_02_scorecard.png', bbox_inches='tight')
plt.show()

print()
print("SCORING NOTE: This scorecard weights growth and risk equally.")
print("For IRDAI regulatory purposes, Claims Ratio weight may need increasing")
print("if consumer protection is the primary mandate.")
print("Confidence: MEDIUM (3-yr CAGR sensitive to COVID base effect in FY20)")


In [ ]:
# ── KPI 6: State Coverage Equity Index ───────────────────────────────────
# Construct a composite equity score per state:
# 1. Premium per capita proxy (Premium / state area as rank proxy — true population not in dataset)
# 2. 3-yr growth rate
# 3. Absolute premium rank

df71_wide = df71.pivot_table(index='state', columns=['year','metric'], values='value').reset_index()

# Use 2021-22 premium as primary metric
df71_prem_22 = df71[
    (df71.year=='2021-22') & (df71.metric=='Grp_Premium_Lakh')
].groupby('state')['value'].sum().reset_index()
df71_prem_22.columns = ['State','Premium_Lakh_2022']

df71_prem_19 = df71[
    (df71.year=='2019-20') & (df71.metric=='Grp_Premium_Lakh')
].groupby('state')['value'].sum().reset_index()
df71_prem_19.columns = ['State','Premium_Lakh_2019']

state_eq = df71_prem_22.merge(df71_prem_19, on='State', how='left')
state_eq['Growth_2yr_Pct'] = (state_eq['Premium_Lakh_2022']/state_eq['Premium_Lakh_2019'] - 1)*100
state_eq['Prem_Rank'] = state_eq['Premium_Lakh_2022'].rank(ascending=True)
state_eq['Growth_Rank'] = state_eq['Growth_2yr_Pct'].rank(ascending=True)

state_eq['Equity_Score'] = (0.5 * state_eq['Prem_Rank'] + 0.5 * state_eq['Growth_Rank'])
state_eq['Equity_Score'] = (state_eq['Equity_Score'] / state_eq['Equity_Score'].max()) * 100

state_eq_sorted = state_eq.sort_values('Equity_Score', ascending=False).dropna()

fig, ax = plt.subplots(figsize=(12, 9))
colors_eq = ['#1f4e79' if s >= 70 else '#2e75b6' if s >= 40 else '#e05c5c' 
             for s in state_eq_sorted['Equity_Score']]
ax.barh(state_eq_sorted['State'], state_eq_sorted['Equity_Score'],
        color=colors_eq, edgecolor='white', linewidth=0.4)
ax.axvline(70, color=C_GREEN, linestyle='--', alpha=0.7, label='High equity zone (>70)')
ax.axvline(40, color='orange', linestyle='--', alpha=0.7, label='Moderate zone (40-70)')
ax.set_title('State-Level Health Insurance Equity Score\n(Composite: Premium Level + Growth Momentum)', fontweight='bold')
ax.set_xlabel('Equity Score (0-100)')
ax.legend(fontsize=9)
ax.tick_params(axis='y', labelsize=8)
plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/kpi_03_state_equity.png', bbox_inches='tight')
plt.show()

low_equity = state_eq_sorted[state_eq_sorted['Equity_Score'] < 40]['State'].tolist()
print(f"Low equity states (<40 score): {low_equity}")
print()
print("INSIGHT: States in the low-equity tier require targeted policy intervention —")
print("both incentivizing insurer entry and subsidized product design.")


## 4.2 Dashboard Product Specification

### Core KPI Tiles (always-visible)
| Tile | Metric | Calculation | Frequency |
|------|--------|-------------|-----------|
| Market Premium | Total Gross Premium (₹Cr) | Sum all insurers | Annual |
| Coverage | Total Persons Covered (Mn) | Sum all insurers × 1000 | Annual |
| Market Claims Ratio | Aggregate | Net Claims / Net Earned Premium | Annual |
| HHI | Market concentration | Σ(share²) × 10000 | Annual |
| YoY Growth | Premium growth | (Current - Prior) / Prior | Annual |

### Secondary Diagnostic Panels
1. Sector share waterfall (Public / Private / Standalone)
2. Insurer-level scorecard table with tier coloring
3. State equity map (choropleth)
4. Claims ratio by segment heatmap

### Filters
- Fiscal Year (single or range)
- Insurer Type (Public / Private / Standalone / All)
- Segment (Govt / Group / Individual / Family Floater / All)
- State / Region

### Drill-Down Paths
- Market → Sector → Insurer → Year
- State → District (if data becomes available)
- Segment → Product type → Insurer

### Update Frequency
- Annual (aligned with IRDAI Handbook publication)
- Can be automated once raw Excel input format is standardized

### Power BI / Tableau Replication Notes
- All KPIs are additive (sum aggregatable) except Claims Ratio (requires numerator/denominator carry)
- Use CALCULATE with DIVIDE for claims ratio in DAX
- State equity score requires custom measure; pre-compute and load as static table
